<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

In [1]:
# TODO 0: 실습을 위해 아래 패키지를 import 해주세요.
import numpy as np
import seaborn as sns
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

<style>
table { width: 100% !important; }
th, td { text-align: left !important; }
</style>

# 학습 내용
>이번 장에서는 <strong>정규방정식, 최소제곱법, SVD(Normal Equation, Least Squares, SVD)</strong>에 대해 학습합니다.</br></br>
>선형대수 기반의 해석적 해법으로 선형 회귀 문제를 학습해봅시다.

</br>

# 정규방정식 (Normal Equation)
> 선형 회귀의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">최적 파라미터를 한 번에 계산</mark>하는 해석적(closed-form) 해법입니다.

>선형 회귀 모델은 다음과 같이 표현됩니다.
>$$X\theta = y$$
>$X$는 일반적으로 **정사각행렬이 아닙니다.**</br>
>따라서 $X$의 역행렬이 존재하지 않아, 양변에 $X^{-1}$를 곱하는 방식으로는 $\theta$를 구할 수 없습니다.</br>
>양변에 $X^T$를 곱하면 $(n \times n)$ **정사각행렬**을 만들 수 있습니다.
>$$X^T X \theta = X^T y$$
>$X^TX$가 역행렬을 가지면 양변에 $(X^TX)^{-1}$을 곱하여 $\theta$를 구합니다.
>$$(X^T X)^{-1} X^T X \theta = (X^T X)^{-1} X^T y$$
>$$I \theta = (X^T X)^{-1} X^T y$$
>$$\theta = (X^T X)^{-1} X^T y$$

In [6]:
# TODO 1: diamonds 데이터셋에서 carat을 X, price를 y로 저장해봅시다. (처음 100개 사용)

# X 는 원래 데이터, 세타는 가중치값,
# 정사각행렬을 만들어줘서 역행렬을 곱해주면 세타 = X^-1 * Y 가 된다
# X^T 는 전치
# 정사각 행렬이 아닌애드은 전치를 곱해주면 정사각행렬이 됨

df = sns.load_dataset("diamonds")

X = df[["carat"]].values[:100]
Y = df["price"].values[:100]

print(X[:5])
print(Y[:5])

print(df)

[[0.23]
 [0.21]
 [0.23]
 [0.29]
 [0.31]]
[326 326 327 334 335]
       carat        cut color clarity  depth  table  price     x     y     z
0       0.23      Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
1       0.21    Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
2       0.23       Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
3       0.29    Premium     I     VS2   62.4   58.0    334  4.20  4.23  2.63
4       0.31       Good     J     SI2   63.3   58.0    335  4.34  4.35  2.75
...      ...        ...   ...     ...    ...    ...    ...   ...   ...   ...
53935   0.72      Ideal     D     SI1   60.8   57.0   2757  5.75  5.76  3.50
53936   0.72       Good     D     SI1   63.1   55.0   2757  5.69  5.75  3.61
53937   0.70  Very Good     D     SI1   62.8   60.0   2757  5.66  5.68  3.56
53938   0.86    Premium     H     SI2   61.0   58.0   2757  6.15  6.12  3.74
53939   0.75      Ideal     D     SI2   62.2   55.0   2757  5.83  5.87  3.64

[53940 rows 

In [ ]:
# TODO 2: X_b에 편향 1을 추가하여 저장해봅시다.

X_b = np.c_[np.ones((len(X),1)), X] # 계산하기 위해 채우는거임 ones

print(X_b.shape)

print(X_b)

(100, 2)
[[1.   0.23]
 [1.   0.21]
 [1.   0.23]
 [1.   0.29]
 [1.   0.31]
 [1.   0.24]
 [1.   0.24]
 [1.   0.26]
 [1.   0.22]
 [1.   0.23]
 [1.   0.3 ]
 [1.   0.23]
 [1.   0.22]
 [1.   0.31]
 [1.   0.2 ]
 [1.   0.32]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.23]
 [1.   0.23]
 [1.   0.31]
 [1.   0.31]
 [1.   0.23]
 [1.   0.24]
 [1.   0.3 ]
 [1.   0.23]
 [1.   0.23]
 [1.   0.23]
 [1.   0.23]
 [1.   0.23]
 [1.   0.23]
 [1.   0.23]
 [1.   0.23]
 [1.   0.23]
 [1.   0.31]
 [1.   0.26]
 [1.   0.33]
 [1.   0.33]
 [1.   0.33]
 [1.   0.26]
 [1.   0.26]
 [1.   0.32]
 [1.   0.29]
 [1.   0.32]
 [1.   0.32]
 [1.   0.25]
 [1.   0.29]
 [1.   0.24]
 [1.   0.23]
 [1.   0.32]
 [1.   0.22]
 [1.   0.22]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.35]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.42]
 [1.   0.28]
 [1.   0.32]
 [1.   0.31]
 [1.   0.31]
 [1.   0.24]
 [1.   0.24]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.3 ]
 [1.   0.26]
 [1

💡`np.c_`란?
> `np.c_`는 열(column) 방향으로 배열을 결합하는 NumPy 인덱서입니다.</br>
>
> ```python
> np.c_[np.ones((3, 1)), np.array([[1], [2], [3]])]
> # array([[1., 1.],
> #        [1., 2.],
> #        [1., 3.]])
> ```
> 선형 회귀에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">편향(bias) 항을 추가</mark>할 때 자주 사용됩니다.

💡`np.ones`란?
> 모든 원소가 1인 배열을 생성하는 NumPy 함수입니다.</br>
> 인자로 배열의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">shape(형상)</mark>을 전달합니다.
>
> ```python
> np.ones((3, 1))
> # array([[1.],
> #        [1.],
> #        [1.]])
>
> np.ones(5)
> # array([1., 1., 1., 1., 1.])
> ```
> 선형 회귀에서 편향(bias) 열을 만들 때 `np.ones((샘플 수, 1))` 형태로 사용합니다.

In [11]:
# TODO 3: 정규방정식을 이용하여 기울기와 절편을 구해봅시다.

# (X^T X)^{-1} X^T y
# 행렬 곱은 @
# np.linalg.inv <- 역행렬 구하기
theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ Y

print(theta)




[-731.06400312 4329.8853521 ]


💡정규방정식에 사용된 NumPy 연산

| 연산 | 문법 | 설명 | 예시 |
|------|------|------|------|
| **전치 (Transpose)** | `A.T` | 행과 열을 뒤집습니다. $(m \times n) \rightarrow (n \times m)$ | `X_b.T` → (2,5) 행렬 |
| **행렬 곱 (Matrix Multiply)** | `A @ B` | 두 행렬의 곱을 계산합니다. `np.dot(A, B)`와 동일합니다. | `X_b.T @ X_b` → (2,2) 행렬 |
| **역행렬 (Inverse)** | `np.linalg.inv(A)` | 정방행렬 $A$의 역행렬 $A^{-1}$을 구합니다. $A A^{-1} = I$ | `np.linalg.inv(X_b.T @ X_b)` |

💡정규방정식의 한계
> $X^TX$의 역행렬이 존재해야 하며, 피처 수가 많으면(d > 10,000) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">계산 비용이 $O(d^3)$</mark>으로 비효율적입니다.

</br>

# SVD (Singular Value Decomposition)
> 임의의 행렬을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">세 행렬의 곱으로 분해</mark>하는 선형대수 기법입니다.

$$A = U \Sigma V^T$$

<div style="text-align:center">

</div>

In [15]:
# TODO 4: X_b를 SVD로 분해한 뒤, 분해된 행렬의 곱으로 원본 복원을 검증해봅시다.
# 역행렬을 구할 수가 없는 경우가 있음
# 하나의 행렬을 3개로 쪼개서 u시그마v.T 로 응용하기가 가능.. 외우자

# linalg.svd
U, singular_values, V_transpose = np.linalg.svd(X_b, full_matrices=False)

print(U)
print('---------------------')
print(singular_values)
print('---------------------')
print(V_transpose)

# 복원 검증
X_b_reconstructed = U @ np.diag(singular_values) @ V_transpose

print(X_b_reconstructed)


[[-9.71337164e-02 -6.45307513e-02]
 [-9.65381664e-02 -7.74605466e-02]
 [-9.71337164e-02 -6.45307513e-02]
 [-9.89203666e-02 -2.57413656e-02]
 [-9.95159167e-02 -1.28115704e-02]
 [-9.74314915e-02 -5.80658537e-02]
 [-9.74314915e-02 -5.80658537e-02]
 [-9.80270415e-02 -4.51360585e-02]
 [-9.68359414e-02 -7.09956490e-02]
 [-9.71337164e-02 -6.45307513e-02]
 [-9.92181417e-02 -1.92764680e-02]
 [-9.71337164e-02 -6.45307513e-02]
 [-9.68359414e-02 -7.09956490e-02]
 [-9.95159167e-02 -1.28115704e-02]
 [-9.62403913e-02 -8.39254442e-02]
 [-9.98136917e-02 -6.34667276e-03]
 [-9.92181417e-02 -1.92764680e-02]
 [-9.92181417e-02 -1.92764680e-02]
 [-9.92181417e-02 -1.92764680e-02]
 [-9.92181417e-02 -1.92764680e-02]
 [-9.92181417e-02 -1.92764680e-02]
 [-9.71337164e-02 -6.45307513e-02]
 [-9.71337164e-02 -6.45307513e-02]
 [-9.95159167e-02 -1.28115704e-02]
 [-9.95159167e-02 -1.28115704e-02]
 [-9.71337164e-02 -6.45307513e-02]
 [-9.74314915e-02 -5.80658537e-02]
 [-9.92181417e-02 -1.92764680e-02]
 [-9.71337164e-02 -6

💡SVD가 왜 중요한가?
> SVD는 최소제곱법, PCA, 추천 시스템 등 다양한 분야의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">핵심 수학 도구</mark>입니다.</br>
> 특이값(Singular Value)의 크기로 데이터의 중요한 차원을 판별합니다.

</br>

# 유사 역행렬 (Pseudo-Inverse)
> 역행렬이 존재하지 않는 행렬에 대해서도 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">SVD를 이용하여 역행렬과 유사한 행렬</mark>을 구할 수 있습니다.

$$A^+ = V \Sigma^+ U^T$$

> $\Sigma^+$는 $\Sigma$의 0이 아닌 특이값을 역수로 바꾼 행렬입니다.

### 왜 유사 역행렬이 필요한가?

정규방정식 $\theta = (X^TX)^{-1}X^Ty$ 에서 $X^TX$의 **역행렬이 존재하지 않는 경우**가 있습니다.

> 예시: **다중공선성(Multicollinearity)** — 피처 간에 선형 관계가 존재할 때

$$
X = \begin{bmatrix} 1 & 2 \\ 2 & 4 \\ 3 & 6 \end{bmatrix}
\quad \Rightarrow \quad
X^TX = \begin{bmatrix} 14 & 28 \\ 28 & 56 \end{bmatrix}
$$

> 두 번째 열이 첫 번째 열의 2배이므로 $\det(X^TX) = 14 \times 56 - 28 \times 28 = 0$</br>
> 행렬식이 0이면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">역행렬이 존재하지 않아</mark> 정규방정식을 풀 수 없습니다.</br>
> 이런 경우에도 유사 역행렬 $X^+$는 SVD를 통해 구할 수 있으므로, $\theta = X^+ y$로 해를 구합니다.

<div style="text-align:center">

</div>

In [18]:
# TODO 5: np.linalg.pinv()로 유사 역행렬을 구하고, 정규방정식 결과와 비교해봅시다.

X_b_pinv = np.linalg.pinv(X_b)
theta_pinv = X_b_pinv @ Y

print(theta)

print('------------')
print(theta_pinv)

[-731.06400312 4329.8853521 ]
------------
[-731.06400312 4329.8853521 ]


💡유사 역행렬 vs 역행렬
> `np.linalg.inv()`는 정방행렬이고 역행렬이 존재할 때만 사용 가능합니다.</br>
> `np.linalg.pinv()`는 내부적으로 SVD를 사용하므로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">어떤 크기의 행렬에도 동작</mark>합니다.</br>
> 정방행렬이면서 역행렬이 존재하는 경우, 유사 역행렬 = 역행렬과 동일한 결과를 줍니다.

</br>

# 최소제곱법 (Least Squares)
> `np.linalg.lstsq`는 정규방정식보다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">수치적으로 안정적</mark>인 최소제곱 해법입니다.</br>
> 내부적으로 SVD + 유사 역행렬을 사용하여 $\theta = X^+ y$를 구합니다.

💡`np.linalg.lstsq`의 반환값

| 반환값 | 변수명 | 설명 |
|--------|--------|------|
| **theta** | `theta_least_squares` | 최소제곱 해. $\theta = X^+ y$로 구한 절편과 기울기 벡터 |
| **잔차 (residuals)** | `residuals` | 잔차 제곱합 $\sum(y_i - \hat{y}_i)^2$. 모델 예측값과 실제값 사이의 오차를 나타내며, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">값이 작을수록 모델이 데이터를 잘 설명</mark>합니다. |
| **랭크 (rank)** | `matrix_rank` | 행렬 $X$의 유효 랭크. 선형 독립인 열의 수를 의미하며, 랭크가 피처 수보다 작으면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">다중공선성</mark>이 존재합니다. |
| **특이값 (singular values)** | `singular_values` | SVD로 구한 $X$의 특이값 배열. 특이값이 0에 가까우면 해당 차원의 정보가 거의 없다는 뜻이며, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">수치적 안정성 판단</mark>에 사용됩니다. |

In [21]:
# TODO 6: 최소제곱법으로 theta, 잔차, 랭크, 특이값을 구하고, 예측값을 출력해봅시다.

theta_least_squares, residuals, matrix_rank, singular_values = np.linalg.lstsq(X_b, Y, rcond=None)

print(theta)

[-731.06400312 4329.8853521 ]


💡lstsq vs 정규방정식
> `lstsq`는 내부적으로 SVD를 사용하여 특이 행렬(singular matrix)에도 동작합니다.</br>
> 실무에서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정규방정식 대신 lstsq를 권장</mark>합니다.